# Aralin 01 - Panimula sa mga AI Agent

Maligayang pagdating sa unang aralin sa kurso na **AI Agents para sa mga Baguhan**!

Ang isang **AI agent** ay isang programa na gumagamit ng malaking language model (LLM) bilang makina ng pangangatwiran at maaaring gumawa ng *mga aksyon* sa totoong mundo — tumawag ng mga API, mag-query sa mga database, o magpatakbo ng code — upang makamit ang isang layunin para sa isang user.

Sa notebook na ito gagawa ka ng iyong unang agent: isang **Travel Agent** na nagrerekomenda ng mga destinasyon sa bakasyon. Kasabay nito matututuhan mo kung paano:

1. Kumonekta sa Microsoft Foundry Agent Service gamit ang **Microsoft Agent Framework**.
2. Bigyan ang agent ng isang **tool** — isang simpleng Python function na maaari nitong tawagan.
3. Patakbuhin ang agent at suriin ang tugon nito.
4. I-stream ang tugon ng agent token-by-token.


## Setup

Bago patakbuhin ang notebook na ito, siguraduhing mayroon ka:

1. **Isang Microsoft Foundry project** na may na-deploy na chat model (hal. `gpt-5-mini`).
2. **Nakalog-in gamit ang Azure CLI** — patakbuhin ang `az login` sa iyong terminal.
3. **Naitakda ang mga kinakailangang environment variables:**
   - `AZURE_AI_PROJECT_ENDPOINT` — ang endpoint ng iyong Microsoft Foundry project.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ang pangalan ng iyong na-deploy na modelo.

Ina-install ng cell sa ibaba ang mga Python package na kailangan mo.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Paglikha ng Iyong Unang Ahente

Ang isang ahente ay nangangailangan ng dalawang bagay:

- **Mga Tagubilin** na nagsasabi kung *sino siya* at *paano kumilos* (isang system prompt).
- **Mga Kasangkapan** — Mga function ng Python na may dekorasyong `@tool` na maaaring tawagin ng ahente upang kumuha ng impormasyon o magsagawa ng mga aksyon.

Sa ibaba ay nagtatakda tayo ng isang simpleng kasangkapan na nagbabalik ng listahan ng mga sikat na destinasyon ng bakasyon. Gagamitin ng ahente ang kasangkapang ito kapag may humiling ng rekomendasyon sa paglalakbay.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Mga Streaming na Tugon

Para sa mas interaktibong karanasan maaari mong **i-stream** ang tugon ng ahente. Sa halip na hintayin ang buong sagot, ang ahente ay nagbibigay ng mga piraso ng teksto habang ito ay nililikha. Ito ay lalong kapaki-pakinabang sa mga chat interface kung saan gusto mong ipakita ang output nang real time.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Buod

Sa araling ito natutunan mo kung paano:

- **Lumikha ng provider** na kumokonekta sa Microsoft Foundry Agent Service gamit ang `FoundryChatClient`.
- **Magdeklara ng tool** gamit ang dekorator na `@tool` upang magamit ng agent ang iyong mga function sa Python.
- **Patakbuhin ang agent** gamit ang mensahe ng gumagamit at iprinta ang tugon nito.
- **I-stream ang mga tugon** para sa real-time na output.

Sa susunod na aralin, mas susuriin natin ang mga agentic frameworks at malalaman kung paano bigyan ang mga agent ng mas makapangyarihang tools at kakayahang mag-isip ng multi-step reasoning.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
